In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from dotenv import load_dotenv, find_dotenv
from utils.outlier_detection import detect_outliers_by_series
from utils.formatting_functions import format_historical_kpi_data, format_simulator_data
from utils.db_functions import get_connection_string, get_latest_sim_data, get_latest_hist_data

In [35]:
# Data path
base_path = Path('C:/Users/User/OneDrive/Desktop/UN')
data_path = Path('C:/Users/alfon/Desktop/UN/Data')
#data_path = Path('C:/Users/GuestUser/Desktop/UN/Data')


data_files_dict = {'index_1030' : ['index_1030 Gender Development Index.xlsx', '2025_metadata_Gender Development Index 2025.xlsx'],
                   'index_9' : ['index_9 EGDI.xlsx', '20251102_metadata_E-Government Development Index_2024.xlsx', 'raw_EGDI_2024.xlsx'],
                   'index_1032': ['20260710_historical data_index_1032_SDG 2026.xlsx', '20260710_metadata_Sustainable_Development_Index_2026.xlsx', 'raw_SDG_2026.xlsx'],

                    # Somalia indices
                   'index_12' : ['index_12 E-participation Index.xlsx', '', 'raw_EPI_2024.xlsx'],
                   'index_1049' : ['index_1049.xlsx', '', 'raw_IIAG_2024.xlsx'],
                   'index_1037' : ['index_1037.xlsx', '', 'raw_GOVTECH_2022.xlsx'],
                   'index_1022': ['index_1022 Women Peace and security Index.xlsx','','raw_WPS_2025.xlsx'],
                   
                   # Somalia second batch
                   'index_1050': ['index_1050 Global Hunger.xlsx','','raw_GHI.xlsx'],
                   'index_5': ['index_5 ICT Developement Index.xlsx','','raw_ITU_2024.xlsx'],
                   'index_1036': ['20262205_historical data_index_1036_WPF_2026.xlsx','','raw_WPF_2026.xlsx'],

                   # Third
                   'index_1027': ['index_1027 Human Development Index.xlsx','','raw_HDI.xlsx'],
                   'index_1030': ['index_1030 Gender Development Index.xlsx','','raw_GDI_2024.xlsx'],
                   'index_1031': ['index_1031 Planetary adjusted HDI.xlsx','','raw_PPHDI.xlsx'],
                   'index_1035': ['20260303_Historical data _WBL2.0_2026.xlsx','','raw_WBL2.0_2026.xlsx'],
                   # None,
                   'index_1026': ['index_1026 Open data Policies.xlsx','','raw_ODP_2022.xlsx'],
                   'index_1053': ['20260721_historical data_index_1053_BTI2026.xlsx','','raw_BTI_2026.xlsx'],
                   'index_1021': ['index_1021.xlsx','','raw_WBL 1.0.xlsx'],
                   'index_1052': ['20260717_historical data_index_1052_WPJ2026.xlsx','','raw_WJP_2025.xlsx'],
                   'index_1034': ['index_1034.xlsx','','raw_EnviPI_2026.xlsx'],

                   # Check below on Outlier detection
                   'index_1024': ['index_1024_FII.xlsx','','raw_FII_2025 1.xlsx'], # No Somalia

                   'index_1051': ['20263007_historical data_LNOB_index_1051_2026 1.xlsx','','raw_LNOB_2026 1.xlsx'],

                   'index_3': ['index_3.xlsx','','raw_GCYB_2024.xlsx'],
                   'index_7': ['index_7 GII.xlsx','','raw_GII_2025.xlsx'],   # No Somalia
                   'index_10': ['index_10 DARE.xlsx', '', 'raw_DARE 1.xlsx']
                   
                  }
#Human Development Index (index_1027)
#Gender Development Index (index_1030)
#Planetary adjusted Human development Index (index_1031)
#Women Business and the Law 2.0 (index_1035)

index_name = 'index_1052'
index_data_files = data_files_dict[index_name]

# Read straight from DB
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found in the current directory or its parents")
load_dotenv(env_path, override=True)
read_from_db = True
db_env = 'prod'

only_escwa_countries = False # Run on all/escwa countries

index_historical_kpi_data_path = Path(data_path) / 'Historical' / index_data_files[0]
#index_metadata_path = Path(data_path) / 'Metadata' / index_data_files[1]
index_simulator_data_path = Path(data_path) / 'Simulator Data' / index_data_files[2]
diagnosis_assessment_template_path = Path(data_path) / 'Reference templates' / 'Data_Diagnosis_Assessment_template.xlsx'

# Reference template
diagnosis_assessment_template = pd.read_excel(diagnosis_assessment_template_path)

# Read raw data
historical_kpi_data_raw = pd.read_excel(index_historical_kpi_data_path)
#index_metadata_raw = pd.read_excel(index_metadata_path)
#index_metadata_raw['Arab Countries'] = index_metadata_raw['Arab Countries'].str.strip()
simulator_data_raw = pd.read_excel(index_simulator_data_path)

country_iso = pd.read_excel(Path(data_path) / 'metadata_SDG_2026.xlsx', sheet_name='country')
country_iso['country'] = country_iso['country'].str.strip()
country_iso['ISO'] = country_iso['ISO'].str.strip()

# Reference Data
escwa_members = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                 'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']

kpi_key = (simulator_data_raw.iloc[0]
           .rename('Series Name')
           .rename_axis('KPI ID')
           .reset_index()
           .dropna(subset=['Series Name']))
kpi_key['Series Name'] = kpi_key['Series Name'].astype(str).str.strip()
kpi_key['KPI ID'] = kpi_key['KPI ID'].astype(str).str.strip()

kpi_list = kpi_key['KPI ID'].unique().tolist()

In [36]:

connection_string = get_connection_string(env=db_env)
# Simulator Data
simulator_data_db = get_latest_sim_data(connection_string, kpi_list)
simulator_data_db = simulator_data_db.add_suffix('_db')

# Historical KPI Data
historical_kpi_data_long_db = get_latest_hist_data(connection_string, kpi_list)
historical_kpi_data_long_db['data_historical_automated'] = pd.to_numeric(
    historical_kpi_data_long_db['data_historical_automated'], errors='coerce').astype('Float64')
historical_kpi_data_long_db['KPI ID'] = historical_kpi_data_long_db['KPI ID'].astype(str).str.strip()
historical_kpi_data_long_db['year_historical_automated'] = pd.to_numeric(
    historical_kpi_data_long_db['year_historical_automated'], errors='coerce').astype('Int64')
historical_kpi_data_long_db = historical_kpi_data_long_db.add_suffix('_db')


simulator_data_excel = format_simulator_data(simulator_data_raw, kpi_key,
                                       pivot_names=('Country', 'ISO', 'Region', 'Sub-region', 'Income Type'))
simulator_data_excel = simulator_data_excel.add_suffix('_excel')

historical_kpi_data_long_excel = format_historical_kpi_data(historical_kpi_data_raw, iso_filter=None)
historical_kpi_data_long_excel = historical_kpi_data_long_excel.add_suffix('_excel')

Simulator Comparison

In [37]:
simulator_comparison_df = simulator_data_excel.merge(simulator_data_db, how='left',
                                                     left_on=['KPI ID_excel', 'ISO_excel'],
                                                     right_on=['KPI ID_db', 'ISO_db'])
simulator_comparison_df['data_diff'] = abs(simulator_comparison_df['data_report_excel']-simulator_comparison_df['data_report_db'])
simulator_comparison_df['data_equal'] = simulator_comparison_df['data_diff'] <= 0.00001
simulator_comparison_df['both_na'] = simulator_comparison_df['data_report_excel'].isna() & simulator_comparison_df['data_report_db'].isna()
simulator_comparison_df['no_discrepancy'] = simulator_comparison_df['data_equal'] | simulator_comparison_df['both_na']
discrepancy = simulator_comparison_df[~simulator_comparison_df['no_discrepancy']]
#discrepancy

# Indices to look at
# - 1024


In [38]:
discrepancy

,Country_excel,ISO_excel,Series Name_excel,KPI ID_excel,data_report_excel,Country_db,ISO_db,Series Name_db,KPI ID_db,data_report_db,data_report_year_db,data_diff,data_equal,both_na,no_discrepancy


Historical Comparison

In [39]:
historical_comparison_df = historical_kpi_data_long_excel.merge(historical_kpi_data_long_db, how='left',
                                                                left_on=['KPI ID_excel', 'ISO_excel', 'year_historical_automated_excel'],
                                                                right_on=['KPI ID_db', 'ISO_db', 'year_historical_automated_db'])
historical_comparison_df['data_diff'] = abs(historical_comparison_df['data_historical_automated_excel']-historical_comparison_df['data_historical_automated_db'])
historical_comparison_df['data_equal'] = historical_comparison_df['data_diff'] <= 0.00001
historical_comparison_df['both_na'] = historical_comparison_df['data_historical_automated_excel'].isna() & historical_comparison_df['data_historical_automated_db'].isna()
historical_comparison_df['no_discrepancy'] = historical_comparison_df['data_equal'] | historical_comparison_df['both_na']
discrepancy = historical_comparison_df[~historical_comparison_df['no_discrepancy']]

In [40]:
discrepancy

,KPI ID_excel,Series Name_excel,year_historical_automated_excel,ISO_excel,data_historical_automated_excel,KPI ID_db,Series Name_db,ISO_db,year_historical_automated_db,data_historical_automated_db,data_diff,data_equal,both_na,no_discrepancy
13,3624,Government powers are effectively limited by ...,2014,ALB,0.48833,3624,Government powers are effectively limited by t...,ALB,2014,0.334548,0.153782,False,False,False
14,3624,Government powers are effectively limited by ...,2015,ALB,0.552433,3624,Government powers are effectively limited by t...,ALB,2015,0.329095,0.223338,False,False,False
73,3629,Government officials in the executive branch ...,2014,ALB,0.450499,3629,Government officials in the executive branch d...,ALB,2014,0.405009,0.04549,False,False,False
75,3629,Government officials in the executive branch ...,2016,ALB,0.347674,3629,Government officials in the executive branch d...,ALB,2016,0.391386,0.043712,False,False,False
109,3632,Government officials in the legislative branc...,2014,ALB,0.292948,3632,Government officials in the legislative branch...,ALB,2014,0.186631,0.106318,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73557,3636,Complaint mechanisms,2023,GAB,0.500272,3636,Complaint mechanisms,GAB,2023,0.5154,0.015128,False,False,False
73629,3642,Freedom from arbitrary interference with priv...,2023,GAB,0.126471,3642,Freedom from arbitrary interference with priva...,GAB,2023,0.148529,0.022059,False,False,False
73641,3643,Freedom of assembly and association is effect...,2023,GAB,0.471334,3643,Freedom of assembly and association is effecti...,GAB,2023,0.484299,0.012965,False,False,False
73689,3647,People do not resort to violence to redress p...,2023,GAB,0.359132,3647,People do not resort to violence to redress pe...,GAB,2023,0.389834,0.030702,False,False,False
